# EViT Token Fusion Testing & Visualization (ImageNet-100)

This notebook tests the EViT (Efficient Vision Transformer) token fusion functionality and provides visualizations of:
- Which patches are kept vs. fused at each reduction layer
- Token fusion behavior (pruned tokens are merged into a single extra token)
- Token reduction statistics
- Performance vs. accuracy trade-offs
- Comparison between EViT and baseline models

In [ ]:
# GitHub token-based setup removed.
# If running in a fresh Colab runtime, uncomment these lines:
# !git clone https://github.com/Chalhotra/ViT-Token-Economy.git
# %cd ViT-Token-Economy

In [ ]:
!git checkout test-branch

In [ ]:
!pip -q install -r requirements.txt
!pip -q install -e .

In [ ]:
# Import core modules
from src.imagenet_mapping import build_imagenet100_to_1k_map
from src.models import ModelConfig, create_model, shrink_imagenet1k_head_to_imagenet100
from src.data import DataConfig, load_imagenet100_split, build_transform_for_model, apply_timm_preprocess, build_loader
from src.eval import evaluate_accuracy_latency_throughput, compute_gflops
from src.utils import get_device, num_params
from src.test_models.evit import EVITConfig, apply_evit_pruning, BlockEViTAdapter, collect_evit_viz
import torch

# Import visualization libraries
import matplotlib.pyplot as plt 
import numpy as np
import seaborn as sns
from typing import List, Dict
import pandas as pd

# Set matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
device = get_device()
maps = build_imagenet100_to_1k_map()
print(f"Using device: {device}")

## Utility Functions for EViT Visualization

In [ ]:
def extract_evit_fusion_info(model, x):
    """
    Extract token fusion information from EViT blocks.
    Returns list of (layer_idx, kept_indices, complement_indices) and token counts.
    """
    fusion_info = []
    token_counts = []
    
    # Get initial token count
    B, N, C = x.shape
    num_special = 2 if hasattr(model, 'dist_token') and model.dist_token is not None else 1
    token_counts.append(N - num_special)  # exclude special tokens
    
    # Forward through blocks and collect fusion info
    for i, block in enumerate(model.blocks):
        with torch.no_grad():
            x = block(x)
        
        # Check if this is an EViT block
        if isinstance(block, BlockEViTAdapter):
            idx = block.last_idx
            compl = block.last_compl
            
            if idx is not None:
                # idx contains kept indices plus sentinel (-1) for fused token
                kept_tokens = idx[:, :-1]  # Remove the -1 sentinel
                fusion_info.append((i, kept_tokens.cpu(), compl.cpu() if compl is not None else None))
                # Token count after fusion: kept + 1 (fused token)
                token_counts.append(kept_tokens.shape[1] + 1)
            else:
                token_counts.append(token_counts[-1])
        else:
            token_counts.append(token_counts[-1])
    
    return fusion_info, token_counts


def visualize_evit_fusion(image, patch_size, fusion_info, token_counts, model_name):
    """
    Visualize which patches are kept vs. fused in EViT.
    
    Args:
        image: Original PIL image or tensor
        patch_size: Size of patches (e.g., 16)
        fusion_info: List of (layer_idx, kept_indices_tensor, fused_indices_tensor)
        token_counts: List of token counts at each layer
        model_name: Name for the plot title
    """
    # Convert image to numpy if needed
    if isinstance(image, torch.Tensor):
        img_np = image.permute(1, 2, 0).cpu().numpy()
    else:
        img_np = np.array(image)
    
    # Normalize if needed
    if img_np.max() > 1.0:
        img_np = img_np / 255.0
    
    h, w = img_np.shape[:2]
    n_patches_h = h // patch_size
    n_patches_w = w // patch_size
    total_patches = n_patches_h * n_patches_w
    
    # Create visualization
    n_fusion_layers = len(fusion_info)
    fig, axes = plt.subplots(1, n_fusion_layers + 1, figsize=(5 * (n_fusion_layers + 1), 5))
    if n_fusion_layers == 0:
        axes = [axes]
    
    # Show original image
    axes[0].imshow(img_np)
    axes[0].set_title(f'Original Image\n{total_patches} patches')
    axes[0].axis('off')
    
    # Show fusion at each layer
    for idx, (layer_idx, kept_indices, fused_indices) in enumerate(fusion_info):
        ax = axes[idx + 1]
        
        # Create masks for kept and fused patches
        kept_mask = np.zeros((n_patches_h, n_patches_w))
        fused_mask = np.zeros((n_patches_h, n_patches_w))
        
        kept_idx_np = kept_indices[0].numpy()  # Take first batch item
        
        # Mark kept patches
        for ki in kept_idx_np:
            pi = ki % n_patches_h
            pj = ki // n_patches_h
            if pi < n_patches_h and pj < n_patches_w:
                kept_mask[pi, pj] = 1
        
        # Mark fused patches
        if fused_indices is not None:
            fused_idx_np = fused_indices[0].numpy()
            for fi in fused_idx_np:
                pi = fi % n_patches_h
                pj = fi // n_patches_h
                if pi < n_patches_h and pj < n_patches_w:
                    fused_mask[pi, pj] = 1
        
        # Create overlay
        overlay = img_np.copy()
        for i in range(n_patches_h):
            for j in range(n_patches_w):
                y_start, y_end = i * patch_size, (i + 1) * patch_size
                x_start, x_end = j * patch_size, (j + 1) * patch_size
                
                if fused_mask[i, j] == 1:  # Fused patch (orange tint)
                    overlay[y_start:y_end, x_start:x_end, 0] = np.clip(
                        overlay[y_start:y_end, x_start:x_end, 0] * 0.5 + 0.5, 0, 1
                    )
                    overlay[y_start:y_end, x_start:x_end, 1] = overlay[y_start:y_end, x_start:x_end, 1] * 0.5
                    overlay[y_start:y_end, x_start:x_end, 2] = overlay[y_start:y_end, x_start:x_end, 2] * 0.3
        
        kept_count = int(kept_mask.sum())
        fused_count = int(fused_mask.sum())
        keep_rate = kept_count / total_patches
        
        ax.imshow(overlay)
        ax.set_title(
            f'After Layer {layer_idx}\n'
            f'{kept_count} kept + 1 fused ({keep_rate:.1%})\n'
            f'Fused {fused_count} patches into 1 token'
        )
        ax.axis('off')
    
    plt.suptitle(f'{model_name} - EViT Token Fusion Visualization', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()


def plot_evit_token_timeline(token_counts, reduction_locs, model_name):
    """
    Plot how token count changes through layers with EViT fusion.
    Note: EViT keeps top-K tokens and adds 1 fused token.
    """
    layers = list(range(len(token_counts)))
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(layers, token_counts, marker='o', linewidth=2, markersize=8, label='Total tokens')
    
    # Highlight reduction locations
    for loc in reduction_locs:
        if loc < len(token_counts):
            ax.axvline(x=loc, color='red', linestyle='--', alpha=0.5)
            ax.text(loc, max(token_counts) * 0.95, f'Fusion at {loc}', 
                   rotation=90, verticalalignment='top', fontsize=9)
    
    ax.set_xlabel('Layer Index', fontsize=12)
    ax.set_ylabel('Number of Patch Tokens (including fused)', fontsize=12)
    ax.set_title(f'{model_name} - Token Count Through Layers (EViT)', fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    # Add percentage annotations
    initial = token_counts[0]
    for i, count in enumerate(token_counts):
        if i in reduction_locs or i == 0 or i == len(token_counts) - 1:
            pct = count / initial * 100
            ax.annotate(f'{count} ({pct:.1f}%)', 
                       xy=(i, count), 
                       xytext=(0, 10), 
                       textcoords='offset points',
                       ha='center',
                       fontsize=9,
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.5))
    
    plt.tight_layout()
    plt.show()


def compare_evit_configurations(results_list: List[Dict], title: str = "EViT Configuration Comparison"):
    """
    Create comparison plots for different EViT configurations.
    
    Args:
        results_list: List of result dicts with keys: config_name, accuracy, gflops, params_m, latency, throughput
    """
    df = pd.DataFrame(results_list)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Accuracy comparison
    axes[0, 0].bar(range(len(df)), df['acc1'], color='steelblue')
    axes[0, 0].set_xticks(range(len(df)))
    axes[0, 0].set_xticklabels(df['config_name'], rotation=45, ha='right')
    axes[0, 0].set_ylabel('Top-1 Accuracy (%)')
    axes[0, 0].set_title('Accuracy Comparison')
    axes[0, 0].grid(True, alpha=0.3)
    
    # GFLOPs comparison
    axes[0, 1].bar(range(len(df)), df['gflops'], color='coral')
    axes[0, 1].set_xticks(range(len(df)))
    axes[0, 1].set_xticklabels(df['config_name'], rotation=45, ha='right')
    axes[0, 1].set_ylabel('GFLOPs')
    axes[0, 1].set_title('Computational Cost')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Latency comparison
    axes[1, 0].bar(range(len(df)), df['latency_ms'], color='mediumseagreen')
    axes[1, 0].set_xticks(range(len(df)))
    axes[1, 0].set_xticklabels(df['config_name'], rotation=45, ha='right')
    axes[1, 0].set_ylabel('Latency (ms)')
    axes[1, 0].set_title('Inference Latency')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Efficiency plot (Accuracy vs GFLOPs)
    axes[1, 1].scatter(df['gflops'], df['acc1'], s=100, alpha=0.6, c=range(len(df)), cmap='viridis')
    for i, row in df.iterrows():
        axes[1, 1].annotate(row['config_name'], 
                           (row['gflops'], row['acc1']),
                           xytext=(5, 5), 
                           textcoords='offset points',
                           fontsize=8)
    axes[1, 1].set_xlabel('GFLOPs')
    axes[1, 1].set_ylabel('Top-1 Accuracy (%)')
    axes[1, 1].set_title('Efficiency Plot')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Print summary table
    print("\n" + "="*80)
    print(f"{title} - Summary Table")
    print("="*80)
    print(df.to_string(index=False))
    print("="*80 + "\n")

## Test EViT with Different Configurations

In [ ]:
def run_evit_test(
    model_id: str,
    evit_config: EVITConfig,
    config_name: str,
    batch_size: int = 64,
    visualize: bool = True
):
    """
    Run model with specific EViT configuration and optionally visualize.
    """
    print(f"\n{'='*80}")
    print(f"Testing: {config_name}")
    print(f"Model: {model_id}")
    print(f"Reduction locations: {evit_config.reduction_loc}")
    print(f"Keep rates: {evit_config.keep_rate}")
    print(f"{'='*80}\n")
    
    # Create model
    model = create_model(ModelConfig(model_id=model_id, pretrained=True))
    model = shrink_imagenet1k_head_to_imagenet100(model, maps.new_to_old_map, num_classes=100)
    
    # Apply EViT pruning
    model = apply_evit_pruning(model, evit_config)
    model = model.to(device).eval()
    
    # Load data
    ds = load_imagenet100_split(DataConfig(split='validation'))
    transform = build_transform_for_model(model)
    ds_t = apply_timm_preprocess(ds, transform)
    loader = build_loader(ds_t, DataConfig(batch_size=batch_size, split='validation', shuffle=False))
    
    # Evaluate
    metrics = evaluate_accuracy_latency_throughput(model, loader, device)
    
    # Compute GFLOPs
    sample = ds_t[0]['pixel_values'].unsqueeze(0).to(device)
    gflops = compute_gflops(model, sample)
    
    # Visualization
    if visualize and evit_config.enabled:
        # Get a sample image for visualization
        sample_idx = 42  # arbitrary sample
        sample_image = ds[sample_idx]['image']
        sample_tensor = ds_t[sample_idx]['pixel_values'].unsqueeze(0).to(device)
        
        # Extract fusion information
        with torch.no_grad():
            # Embed patches
            x = model.patch_embed(sample_tensor)
            if hasattr(model, 'cls_token'):
                cls_tokens = model.cls_token.expand(sample_tensor.shape[0], -1, -1)
                x = torch.cat((cls_tokens, x), dim=1)
            if hasattr(model, 'pos_embed'):
                x = x + model.pos_embed
            if hasattr(model, 'pos_drop'):
                x = model.pos_drop(x)
            
            # Extract fusion info
            fusion_info, token_counts = extract_evit_fusion_info(model, x)
        
        # Visualize token fusion
        if fusion_info:
            visualize_evit_fusion(
                sample_image, 
                16,  # patch size
                fusion_info, 
                token_counts,
                f"{model_id} - {config_name}"
            )
            
            # Plot token timeline
            plot_evit_token_timeline(
                token_counts,
                list(evit_config.reduction_loc),
                f"{model_id} - {config_name}"
            )
    
    result = {
        'config_name': config_name,
        'model': model_id,
        'params_m': num_params(model) / 1e6,
        'gflops': gflops,
        **metrics
    }
    
    print(f"\nResults for {config_name}:")
    print(f"  Top-1 Accuracy: {metrics['acc1']:.2f}%")
    print(f"  GFLOPs: {gflops:.3f}")
    print(f"  Latency: {metrics['latency_ms']:.2f} ms")
    print(f"  Throughput: {metrics['throughput']:.1f} samples/sec")
    
    return result

## Baseline (No Fusion)

In [ ]:
# Test baseline without EViT
baseline_config = EVITConfig(enabled=False)
baseline_result = run_evit_test(
    model_id='deit_tiny_patch16_224',
    evit_config=baseline_config,
    config_name='Baseline (No Fusion)',
    visualize=False
)

## Sanity Check: Keep Rate = 1.0 (Should Match Baseline)

In [ ]:
# Sanity check: keep_rate = 1.0 should yield same accuracy
sanity_config = EVITConfig(
    enabled=True,
    keep_rate=(1.0,),
    reduction_loc=(3, 6, 9),
    exponentiate_single_keep_rate=False
)
sanity_result = run_evit_test(
    model_id='deit_tiny_patch16_224',
    evit_config=sanity_config,
    config_name='Sanity Check (keep=1.0)',
    visualize=True
)

In [ ]:
explicit_rates = [0.25, 0.5, 0.7, 0.9]

if 'results' not in globals():
    results = [baseline_result, sanity_result]

for r in explicit_rates:
    cfg = EVITConfig(
        enabled=True,
        keep_rate=(r, r, r),
        reduction_loc=(3, 6, 9),
        exponentiate_single_keep_rate=False,
    )
    name = f"EViT keep=[{r:.2f},{r:.2f},{r:.2f}]"

    if not any(x.get('config_name') == name for x in results):
        out = run_evit_test(
            model_id='deit_tiny_patch16_224',
            evit_config=cfg,
            config_name=name,
            visualize=False,
        )
        results.append(out)

compare_evit_configurations(results, title='DeiT-Tiny EViT Fusion Comparison (Explicit Sweeps)')

In [ ]:
# Final combined comparison table for explicit keep-rate sweeps
explicit_names = [
    'EViT keep=[0.25,0.25,0.25]',
    'EViT keep=[0.50,0.50,0.50]',
    'EViT keep=[0.70,0.70,0.70]',
    'EViT keep=[0.90,0.90,0.90]',
]

df_all = pd.DataFrame(results)
df_explicit = df_all[df_all['config_name'].isin(explicit_names)].copy()

if not df_explicit.empty:
    order = {name: i for i, name in enumerate(explicit_names)}
    df_explicit['order'] = df_explicit['config_name'].map(order)
    df_explicit = df_explicit.sort_values('order').drop(columns=['order'])
    cols = [c for c in ['config_name', 'acc1', 'latency_ms', 'throughput', 'gflops', 'params_m'] if c in df_explicit.columns]
    display(df_explicit[cols].reset_index(drop=True))
else:
    print('No explicit sweep results found yet. Run the explicit sweep cell first.')